# Nutrition5k: exploración del dataset

No entrena nada: lee `data/nutrition5k/` (metadatos, splits oficiales y frames) y, en la última parte, las predicciones
de un run de `outputs/`. Responde a:

1. Cuántos platos hay en cada partición (train, validación, test) y cuántos tienen frames.
2. Cómo se distribuyen los cinco objetivos en cada partición.
3. Qué platos tienen etiquetas imposibles o incoherentes.
4. Cuánto pesan esos platos en el PMAE (que divide por la media del split).
5. Dónde se concentra el error de un modelo.
6. Si validación tiene platos casi duplicados de train (escaneos incrementales); el detalle está en
   `05_incremental_scans.ipynb`.

La validación es la misma que usa `train`: `split_validation` sobre los platos de train oficiales, sin los platos de
`excluded_dishes.csv`, apartando grupos de escaneos enteros (10 %, por hash). Hace falta haber ejecutado `preprocess`.

In [ ]:
import csv
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nutrition5k import load_dataset
from nutrition5k.dataset import split_validation
from nutrition5k.preprocess import label_errors

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ROOT = Path("data/nutrition5k")
MODE = "complete"  # modo de frames
TARGETS = {"mass": "g", "calories": "kcal", "fat": "g", "carb": "g", "protein": "g"}
SPLITS = ("train", "val", "test")
SERIES = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, MUTED = "#0b0b0b", "#52514e"
SPLIT_COLORS = dict(zip(SPLITS, SERIES))

In [ ]:
def load_dishes(root):
    """One row per dish of both metadata files, with its ingredients as {ingredient id: grams}."""
    rows, names = [], {}
    for cafe in ("cafe1", "cafe2"):
        with open(root / f"metadata/dish_metadata_{cafe}.csv", newline="") as handle:
            for row in csv.reader(handle):
                ingredients = {row[i]: float(row[i + 2]) for i in range(6, len(row), 7)}
                names.update({row[i]: row[i + 1] for i in range(6, len(row), 7)})
                calories, mass, fat, carb, protein = map(float, row[1:6])
                rows.append({
                    "dish_id": row[0], "cafe": cafe, "mass": mass, "calories": calories,
                    "fat": fat, "carb": carb, "protein": protein, "ingredients": ingredients,
                })
    dishes = pd.DataFrame(rows).set_index("dish_id")
    dishes["time"] = dishes.index.str.removeprefix("dish_").astype(int)  # dish ids are Unix timestamps
    return dishes, names


dishes, INGREDIENT_NAMES = load_dishes(ROOT)

# Splits exactly as `train` builds them (validation is held out before dropping dishes without frames).
dataset = load_dataset(ROOT, full=True, frame_mode=MODE)
train, validation = split_validation(dataset.train, groups=dataset.groups)
dishes["split"] = "none"  # dishes in the metadata but in neither official split
# Official train dishes in excluded_dishes.csv are in neither train nor validation; excluded test dishes stay in test.
dishes.loc[(ROOT / "dish_ids/splits/rgb_train_ids.txt").read_text().split(), "split"] = "excluded"
for name, records in (("train", train), ("val", validation), ("test", dataset.test)):
    dishes.loc[[record.dish_id for record in records], "split"] = name
frames = {record.dish_id: len(record.frame_paths) for record in (*dataset.train, *dataset.test)}
dishes["frames"] = dishes.index.map(frames).fillna(0).astype(int)
FIRST_FRAME = {record.dish_id: record.frame_paths[0] for record in (*dataset.train, *dataset.test) if record.frame_paths}
print(f"{len(dishes)} dishes in the metadata")

## 1. Particiones

`excluded` son los platos de train oficiales que `preprocess` deja fuera (no están en train ni en validación); los
excluidos de test siguen en test. `frames > 0` son los platos que usan los modelos; los que no tienen frames (vídeo no disponible) se descartan en train,
validación y test. La línea base de la media se evalúa sobre todos los platos de test.

In [ ]:
partitions = dishes.assign(with_frames=dishes["frames"] > 0).pivot_table(
    index="split", columns="with_frames", values="mass", aggfunc="size", fill_value=0
).rename(columns={True: "with frames", False: "without frames"})
partitions["total"] = partitions.sum(axis=1)
partitions["share (%)"] = (100 * partitions["total"] / partitions["total"].sum()).round(1)
display(partitions.reindex([*SPLITS, "excluded", "none"]))
display(pd.crosstab(dishes["split"], dishes["cafe"]))

In [ ]:
used = dishes[dishes["frames"] > 0]
fig, ax = plt.subplots(figsize=(7, 3))
# Log scale: a few dishes have very long videos (one has 7475 frames) and would squash the rest.
ax.hist(used["frames"], bins=np.logspace(np.log10(used["frames"].min()), np.log10(used["frames"].max()), 50), color=SERIES[0])
ax.set_xscale("log")
ax.set_xlabel("Frames per dish (all side cameras, 1 of every 5; log)", color=MUTED)
ax.set_ylabel("Dishes", color=MUTED)
ax.set_title("Frames per dish", loc="left", color=INK)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
display(used.groupby("split")["frames"].describe().round(1))


def frames_per_camera(dish):
    """Frame count per side camera (the file names start with camera_<letter>)."""
    folder = FIRST_FRAME[dish].parent
    return pd.Series([path.name.split("_frame_")[0] for path in folder.glob("*.jpeg")]).value_counts().sort_index().to_dict()


longest = used.nlargest(5, "frames")[["split", "frames"]]
longest["per camera"] = longest.index.map(frames_per_camera)
longest

## 2. Distribución de los objetivos por partición

El PMAE divide el MAE por la **media del split**, así que conviene que las medias se parezcan y vigilar los máximos:
un solo plato extremo mueve mucho la media y el error.

In [ ]:
summary = dishes[dishes["split"].isin(SPLITS)].groupby("split")[list(TARGETS)].agg(
    ["mean", "median", "std", lambda values: values.quantile(0.99), "max"]
).rename(columns={"<lambda_0>": "p99"}, level=1)
summary.reindex(SPLITS).round(1).T

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 3.2))
for ax, (target, unit) in zip(axes, TARGETS.items()):
    values = dishes.loc[dishes["split"].isin(SPLITS), target]
    bins = np.logspace(np.log10(max(values[values > 0].min(), 0.1)), np.log10(values.max()), 40)
    for split in SPLITS:
        ax.hist(dishes.loc[dishes["split"] == split, target], bins=bins, density=True, histtype="step",
                linewidth=1.8, color=SPLIT_COLORS[split], label=split)
    ax.set_xscale("log")
    ax.set_xlabel(f"{target} ({unit}, log)", color=MUTED)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Density", color=MUTED)
axes[0].legend(frameon=False)
fig.suptitle("Target distribution per split", x=0.01, ha="left", color=INK)
plt.tight_layout()
plt.show()

### Platos con muy poca masa

Test tiene algo más de platos ligeros que train y validación. La tabla da, por partición, la proporción de platos por
debajo de cada masa; debajo, la primera imagen de cada plato de test con menos de `LIGHT_MASS` gramos, de menor a mayor.
Con tan poca masa, unos gramos de error son un porcentaje grande de error para ese plato (aunque el PMAE divide por la
media del split, no por la masa de cada plato).

In [ ]:
def show_dishes(dish_ids, title, columns=4):
    """First frame of each dish with its labels."""
    dish_ids = [dish for dish in dish_ids if dish in FIRST_FRAME]
    if not dish_ids:
        print("no frames for these dishes in data/")
        return
    rows = -(-len(dish_ids) // columns)
    fig, axes = plt.subplots(rows, columns, figsize=(3.6 * columns, 3.4 * rows), squeeze=False)
    for ax, dish in zip(axes.flat, dish_ids):
        ax.imshow(plt.imread(FIRST_FRAME[dish]))
        labels = dishes.loc[dish]
        ax.set_title(f"{dish} ({labels['split']})", loc="left", color=INK, fontsize=9)
        ax.set_xlabel("\n".join(f"{target}: {labels[target]:.0f} {unit}" for target, unit in TARGETS.items()),
                      color=MUTED, fontsize=8, loc="left")
        ax.set_xticks([]), ax.set_yticks([])
    for ax in axes.flat[len(dish_ids):]:
        ax.axis("off")
    fig.suptitle(title, x=0.01, ha="left", color=INK)
    plt.tight_layout()
    plt.show()

In [ ]:
LIGHT_MASS = 20  # g

labelled = dishes[dishes["split"].isin(SPLITS)]
display(pd.DataFrame({
    f"< {limit} g (%)": labelled.groupby("split")["mass"].apply(lambda mass: 100 * (mass < limit).mean())
    for limit in (10, 20, 30, 50)
}).reindex(SPLITS).round(1))

light = labelled[(labelled["split"] == "test") & (labelled["mass"] < LIGHT_MASS)].sort_values("mass")
print(f"{len(light)} test dishes under {LIGHT_MASS} g ({(light['frames'] == 0).sum()} without frames are not shown)")
show_dishes(list(light.index), f"Test dishes under {LIGHT_MASS} g")

## 3. Platos con etiquetas sospechosas

Reglas de `nutrition5k.preprocess` (las mismas con las que `preprocess` escribe `excluded_dishes.csv`; cada una es
una comprobación física o de coherencia interna de los metadatos, no una opinión):

- la suma de gramos de los ingredientes no coincide con la masa total (fuera de ±10 %);
- más de 9 kcal/g, más densidad energética que la grasa pura, imposible;
- un solo ingrediente pesa más de 1.5 kg (la mediana de un plato entero es ~145 g).

Solo se informa (no excluye): platos con 0 kcal.

Además se muestra la comprobación de Atwater, 9·grasa + 4·carbohidratos + 4·proteína frente a las calorías: es aproximada
(no cuenta alcohol ni fibra), así que solo sirve como señal.

In [ ]:
reasons = pd.Series(label_errors(ROOT), dtype=str).reindex(dishes.index)
checks = pd.DataFrame({
    "mass mismatch": reasons.str.contains("total mass", na=False),
    "> 9 kcal/g": reasons.str.contains("kcal/g", na=False),
    "ingredient > 1500 g": reasons.str.contains("ingredient over", na=False),
    "excluded (any rule)": reasons.notna(),
    "zero calories": dishes["calories"] == 0,
})
atwater = (9 * dishes["fat"] + 4 * dishes["carb"] + 4 * dishes["protein"]) / dishes["calories"]

print("flagged dishes per rule and split:")
display(checks.groupby(dishes["split"]).sum().reindex([*SPLITS, "excluded", "none"]))
print(f"Atwater ratio (dishes with calories): median {atwater.median():.2f}, "
      f"outside [0.7, 1.3]: {((atwater < 0.7) | (atwater > 1.3)).sum()} dishes")

In [ ]:
suspicious = dishes[reasons.notna()].copy()
suspicious["reasons"] = reasons[suspicious.index]
suspicious["ingredient grams"] = suspicious["ingredients"].map(
    lambda ingredients: ", ".join(f"{INGREDIENT_NAMES[key]} {grams:.0f} g" for key, grams in ingredients.items()))
suspicious[["split", "frames", *TARGETS, "reasons", "ingredient grams"]].sort_values("split").round(1)

In [ ]:
show_dishes(list(suspicious.index), "Dishes with suspicious labels")

## 4. Peso de los platos sospechosos en el PMAE

Media de cada split con y sin esos platos, y PMAE de la línea base de la media (media de train sin ellos → todos los
platos) con y sin ellos. Aquí los platos de train excluidos cuentan como train, para ver el efecto de quitarlos. Si la línea base apenas cambia pero un modelo sí, el problema es que el modelo no puede acertar
una etiqueta imposible, no que el split sea más difícil.

In [ ]:
def pmae(predicted, truth):
    """PMAE per target, in %: sum of absolute errors / sum of true values (= MAE / mean)."""
    return (predicted - truth).abs().sum() / truth.sum() * 100


# The excluded official train dishes count as train here, so "all" shows the split as it was before preprocess.
labelled = dishes.replace({"split": {"excluded": "train"}})
labelled = labelled[labelled["split"].isin(SPLITS)]
clean = labelled.drop(index=suspicious.index, errors="ignore")
train_mean = clean.loc[clean["split"] == "train", list(TARGETS)].mean()  # as the baseline: fitted without them
rows = []
for split in SPLITS:
    for name, frame in (("all", labelled), ("without suspicious", clean)):
        truth = frame.loc[frame["split"] == split, list(TARGETS)]
        errors = pmae(pd.DataFrame([train_mean] * len(truth), index=truth.index), truth)
        rows.append({"split": split, "dishes": name, "count": len(truth), **truth.mean().add_prefix("mean_").to_dict(),
                     "baseline mean PMAE": errors.mean()})
pd.DataFrame(rows).set_index(["split", "dishes"]).round(1)

## 5. Dónde se concentra el error de un modelo

Lee `predictions_val.csv` y `predictions.csv` de un run. El error de un plato es la media, sobre los cinco objetivos, de
|error| / media del objetivo en ese split (como el PMAE). Compara la mediana (plato típico) con la cola (peores 5 %),
y recalcula el mean PMAE de test quitando los platos sospechosos o los peores.

In [ ]:
# Cualquier run terminado con los datos preprocesados (`-clean`), así la fila "val" usa la misma validación que
# el resto del notebook. Test incluye los platos excluidos (se evalúan aparte en "without suspicious").
RUN = Path("outputs/mobilenet/full/v3/v3-ft_all-single-h512-224-complete-full-fbn-clean-s0")
TARGET_COLUMNS = {f"total_{target}": target for target in TARGETS}


def dish_errors(path):
    """Predictions, labels and per-dish error (worst first) of one predictions file."""
    predicted = pd.read_csv(path).set_index("dish_id").rename(columns=TARGET_COLUMNS)[list(TARGETS)]
    truth = dishes.loc[predicted.index, list(TARGETS)]
    return predicted, truth, ((predicted - truth).abs() / truth.mean()).mean(axis=1).sort_values(ascending=False)


errors = {}
if (RUN / "predictions.csv").is_file():
    rows = []
    for split, file in (("val", "predictions_val.csv"), ("test", "predictions.csv")):
        predicted, truth, error = dish_errors(RUN / file)
        errors[split] = error
        rows.append({
            "split": split, "dishes": len(error), "mean PMAE": pmae(predicted, truth).mean(),
            "median dish error (%)": 100 * error.median(),
            "worst 5% share of error": error.head(len(error) // 20).sum() / error.sum(),
            "mean PMAE without suspicious": pmae(predicted.drop(index=suspicious.index, errors="ignore"),
                                                 truth.drop(index=suspicious.index, errors="ignore")).mean(),
            "mean PMAE without worst 3": pmae(predicted.drop(index=error.index[:3]), truth.drop(index=error.index[:3])).mean(),
        })
    display(pd.DataFrame(rows).set_index("split").round(2))
else:
    print(f"no predictions in {RUN}")

In [ ]:
if errors:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for split, error in errors.items():
        ax.plot(np.arange(1, len(error) + 1) / len(error) * 100, error.cumsum() / error.sum() * 100,
                color=SPLIT_COLORS[split], linewidth=2, label=split)
    ax.set_xlabel("Worst dishes (% of the split)", color=MUTED)
    ax.set_ylabel("Share of the total error (%)", color=MUTED)
    ax.set_title("How concentrated the error is", loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e6e3", linewidth=0.8)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    show_dishes(list(errors["test"].index[:8]), f"{RUN.name}: test dishes with the largest error")

## 6. Platos casi duplicados entre particiones

Nutrition5k escanea el mismo plato varias veces mientras se le añaden ingredientes, y los ids son marcas de tiempo.
Para cada plato de validación y de test: segundos hasta el plato de train más cercano en el tiempo. Si validación
tiene muchos vecinos a pocos segundos y test no, validación contiene variantes de platos de train y sale optimista.
La detección de grupos y una partición alternativa están en `05_incremental_scans.ipynb`.

In [ ]:
train_times = np.sort(dishes.loc[dishes["split"] == "train", "time"].to_numpy())


def gap_to_train(times):
    position = np.clip(np.searchsorted(train_times, times), 1, len(train_times) - 1)
    return np.minimum(np.abs(times - train_times[position - 1]), np.abs(train_times[position] - times))


gaps = {split: gap_to_train(dishes.loc[dishes["split"] == split, "time"].to_numpy()) for split in ("val", "test")}
display(pd.DataFrame({
    split: {"median gap (s)": np.median(gap), "within 60 s (%)": 100 * (gap <= 60).mean(),
            "within 120 s (%)": 100 * (gap <= 120).mean()}
    for split, gap in gaps.items()
}).T.round(1))

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(0, 6, 50)
for split, gap in gaps.items():
    ax.hist(np.maximum(gap, 1), bins=bins, density=True, histtype="step", linewidth=2, color=SPLIT_COLORS[split], label=split)
ax.set_xscale("log")
ax.set_xlabel("Seconds to the nearest train dish (log)", color=MUTED)
ax.set_ylabel("Density", color=MUTED)
ax.set_title("Time to the nearest train dish", loc="left", color=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()